# Notebook 01: NumPy Foundations for Machine Learning

**Module:** 01 Python Revision  
**Duration:** ~2 hours

---

## Learning Objectives

1. Understand scalars, vectors, and matrices as NumPy arrays
2. Master array creation, indexing, slicing, and broadcasting
3. Perform dot products and matrix multiplication
4. Use axis-based aggregations (sum, mean along rows/columns)
5. Connect arrays to ML data representation

## 1. Intuition: Why NumPy?

Machine Learning is **numerical computation on large arrays**:

- A dataset of 10,000 houses × 8 features = matrix of shape `(10000, 8)`
- A grayscale image 512×512 = matrix of shape `(512, 512)`
- A satellite tile with 6 bands = tensor of shape `(6, 512, 512)`

Python lists are flexible but **slow** for this. NumPy stores data in contiguous memory blocks and runs operations in compiled C code.

**Why ML uses NumPy instead of lists:**
1. **Homogeneous dtype** — all elements same type (float64, int32)
2. **Contiguous memory** — cache-friendly, vectorizable
3. **Broadcasting** — operate on different shapes without loops
4. **Linear algebra** — dot product, matrix inverse built-in
5. **GPU bridge** — PyTorch/TensorFlow tensors extend ndarrays

## 2. Scalars, Vectors, Matrices

| Math | NumPy | Shape | Example |
|------|-------|-------|--------|
| Scalar $c$ | 0-d array | `()` | `np.array(3.14)` |
| Vector $\mathbf{x}$ | 1-d array | `(n,)` | `np.array([1,2,3])` |
| Matrix $X$ | 2-d array | `(m, n)` | `np.array([[1,2],[3,4]])` |
| Tensor | n-d array | `(d1,...,dk)` | image batch `(32, 3, 224, 224)` |

In [ ]:
import numpy as np

# Scalar
scalar = np.array(42)
print(f'Scalar: {scalar}, shape: {scalar.shape}, ndim: {scalar.ndim}')

# Vector
vector = np.array([1.0, 2.0, 3.0, 4.0])
print(f'Vector: {vector}, shape: {vector.shape}')

# Matrix
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])
print(f'Matrix shape: {matrix.shape}')  # (2, 3) = 2 rows, 3 columns
print(matrix)

## 3. Array Creation and dtype

Every element in an ndarray has the **same data type** (`dtype`). ML typically uses `float32` (GPU) or `float64` (CPU training).

In [ ]:
# Common creation methods
zeros = np.zeros((3, 4))       # 3x4 matrix of zeros
ones = np.ones((2, 3))         # 2x3 matrix of ones
identity = np.eye(3)           # 3x3 identity matrix
arange = np.arange(0, 10, 2)   # [0, 2, 4, 6, 8]
linspace = np.linspace(0, 1, 5)  # 5 evenly spaced values [0, 0.25, ..., 1]

rng = np.random.default_rng(42)
random_normal = rng.normal(0, 1, size=(1000,))  # mean=0, std=1

print(f'dtype of zeros: {zeros.dtype}')
print(f'Random sample mean: {random_normal.mean():.3f} (expect ~0)')
print(f'Random sample std:  {random_normal.std():.3f} (expect ~1)')

## 4. Indexing and Slicing

Understanding indexing is critical — bugs from wrong indexing cause silent errors in ML pipelines.

- `arr[i]` — element or row
- `arr[i, j]` — element at row i, column j
- `arr[:, j]` — entire column j
- `arr[i, :]` — entire row i
- `arr[arr > 0]` — boolean (fancy) indexing

In [ ]:
data = np.array([[10, 20, 30],
                 [40, 50, 60],
                 [70, 80, 90]])

print('Element [1,2]:', data[1, 2])      # 60
print('Row 0:', data[0])                  # [10 20 30]
print('Column 1:', data[:, 1])            # [20 50 80]
print('Submatrix top-left 2x2:\n', data[:2, :2])

# Boolean indexing: all elements > 50
print('Elements > 50:', data[data > 50])

## 5. Broadcasting

**Broadcasting** applies operations between arrays of different shapes without explicit loops.

Rules (simplified):
1. Compare shapes from right to left
2. Dimensions are compatible if equal or one is 1
3. Size-1 dimension is stretched (broadcast) to match

Example: subtract mean from each column of a matrix.

In [ ]:
X = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)

column_means = X.mean(axis=0)  # shape (3,)
print('Column means:', column_means)

# Broadcasting: (3,3) - (3,) → (3,3)
X_centered = X - column_means
print('Centered matrix:\n', X_centered)
print('Column means after centering:', X_centered.mean(axis=0))  # ~0

## 6. Dot Product and Matrix Multiplication

These are the **most important operations in ML**.

**Dot product** (vectors): $\mathbf{a} \cdot \mathbf{b} = \sum_{i} a_i b_i$

**Matrix multiplication**: $(AB)_{ij} = \sum_k A_{ik} B_{kj}$

In NumPy:
- `np.dot(a, b)` or `a @ b`
- For 2D arrays, `@` is matrix multiply
- For 1D arrays, `@` is dot product

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Dot product
print('a · b =', np.dot(a, b))   # 1*4 + 2*5 + 3*6 = 32
print('a @ b =', a @ b)

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print('A @ B =\n', A @ B)
# [[1*5+2*7, 1*6+2*8], [3*5+4*7, 3*6+4*8]] = [[19,22],[43,50]]

# Connection to Linear Regression (Module 03):
# y_pred = X @ weights + bias

## 7. Axis Operations

The `axis` parameter is a common source of confusion.

For matrix shape `(m, n)`:
- `axis=0` → collapse rows → result shape `(n,)` (column-wise operation)
- `axis=1` → collapse columns → result shape `(m,)` (row-wise operation)

**Memory trick:** `axis=0` goes **down** the rows (operates on each column).

In [ ]:
X = np.array([[1, 2, 3],
              [4, 5, 6]])

print('Shape:', X.shape)
print('Sum all:       ', X.sum())
print('Sum axis=0 (col):', X.sum(axis=0))  # [5, 7, 9]
print('Sum axis=1 (row):', X.sum(axis=1))  # [6, 15]
print('Mean axis=0:   ', X.mean(axis=0))
print('Std axis=0:    ', X.std(axis=0))

## 8. Reshape, Transpose, and Memory

ML constantly reshapes data:
- Flatten image: `(512, 512)` → `(262144,)`
- Batch images: `(32, 3, 224, 224)`
- Feature matrix: `(n_samples, n_features)`

In [ ]:
img = np.arange(12).reshape(3, 4)  # simulate 3x4 image
print('Original:\n', img)
print('Flatten:', img.flatten())
print('Transpose:\n', img.T)

# ML convention: samples as ROWS, features as COLUMNS
n_samples, n_features = 100, 8
X = rng.normal(0, 1, (n_samples, n_features))
print(f'Dataset shape: {X.shape} = ({n_samples} samples, {n_features} features)')

## 9. Connection to GeoSpatial ML

In your **water-bodies-detection** project:

```python
# A Planet tile: 6 spectral bands, 512x512 pixels
tile = np.zeros((6, 512, 512), dtype=np.float32)

# After normalization (percentile scaling per band):
for b in range(6):
    band = tile[b]
    p2, p98 = np.percentile(band[band > 0], [2, 98])
    tile[b] = np.clip((band - p2) / (p98 - p2 + 1e-8), 0, 1)
```

Every raster band is a 2D NumPy array. The full tile is a 3D array. A training batch is a 4D array `(batch, channels, height, width)`.

## 10. Common Mistakes

| Mistake | Problem | Fix |
|---------|---------|-----|
| `a * b` vs `a @ b` | Element-wise vs matrix multiply | Use `@` for linear algebra |
| Wrong axis in mean | Normalizing wrong dimension | Print `.shape` before and after |
| Modifying a view | Unexpected side effects | Use `.copy()` when unsure |
| Integer division | `3/4` in int array truncates | Cast to float: `arr.astype(float)` |
| `(n,) vs (n,1)` shape | sklearn expects 2D targets | Use `.reshape(-1, 1)` |

## Exercise 1: Cosine Similarity

Implement cosine similarity between two vectors **without sklearn**:

$$\text{cosine\_sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

where $\|\mathbf{a}\| = \sqrt{\sum_i a_i^2}$

In [ ]:
# YOUR CODE HERE
def cosine_similarity(a, b):
    """Return cosine similarity between vectors a and b."""
    pass

# Test
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
# Expected: ~0.9746
print('Cosine similarity:', cosine_similarity(u, v))

## Exercise 2: Manual Standardization

Standardize each column of matrix X to zero mean and unit variance:

$$X_{\text{std}} = \frac{X - \mu}{\sigma}$$

Do NOT use sklearn. Use axis operations only.

In [ ]:
# YOUR CODE HERE
X = rng.normal(5, 2, (100, 4))  # 100 samples, 4 features

# Standardize X → X_std
# X_std should have mean ≈ 0 and std ≈ 1 per column


## Interview Questions

1. What is the difference between a Python list and a NumPy array?
2. Explain broadcasting with an example.
3. What does `axis=0` mean for a 2D array operation?
4. Why do we use `float32` instead of `float64` on GPUs?
5. What is the shape of `X @ W` if X is `(100, 10)` and W is `(10, 5)`?

---

## Summary

- NumPy ndarrays are homogeneous, contiguous, fast numerical arrays
- Shape `(m, n)` = m rows, n columns; samples are typically rows
- `@` is matrix multiply; `*` is element-wise
- Broadcasting eliminates loops; axis controls aggregation direction
- Every ML pipeline starts with NumPy arrays

**Next:** [02_Pandas_for_ML.ipynb](02_Pandas_for_ML.ipynb)